# Frobenius Motif Segmentation — v2 (Automatic + Focused IGSM)

Two segmentation modes, selectable per run:

- **Automatic (v1)** — uniform SAM point-grid (same engine as v1 notebook)
- **Focused IGSM** — Otsu binarise → connected components → SAM *point-prompt* per component
- **Compare both** — run both and show side-by-side

**Why IGSM?**  Fuentes-Ferrer et al. (2025) found that automatic-grid SAM produces masks
"mixed between hieroglyphs" on carved stone — the same problem we have on carved wood.
IGSM forces SAM to look only where binarisation already found a dark carved region, which:
- Captures woven/knotwork border patterns as large connected dark components
- Avoids spurious masks from wood-grain texture across the whole panel

**Spatial grid** — detections are mapped onto a reading-order position grid
(top → bottom, left → right) to support future relative-position encoding.

---

```bash
uv run --project src/python jupyter notebook src/python/motif_tuning_v2.ipynb
```

In [15]:
# ── Cell 1: environment setup (Colab only — skip locally) ──────────────────
import sys
ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    print("Colab detected — installing dependencies...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "segment-anything", "opencv-python-headless", "Pillow",
        "numpy>=1.24,<2", "ipywidgets",
    ], check=True)
    import os
    CKPT = "sam_vit_b_01ec64.pth"
    if not os.path.exists(CKPT):
        print("Downloading SAM checkpoint (~370 MB)...")
        subprocess.run(["wget", "-q",
            "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
        ], check=True)
else:
    print("Local — skipping Colab setup.")

Local — skipping Colab setup.


In [16]:
! uv pip install cv2

Using Python 3.11.6 environment at: /Users/korede/code/surulere/african-artifacts/.venv
  × No solution found when resolving dependencies:                                  
  ╰─▶ Because there are no versions of cv2 and you require cv2, we can
      conclude that your requirements are unsatisfiable.


In [ ]:
# ── Cell 2: imports & paths ────────────────────────────────────────────────
import os, sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from PIL import Image
import ipywidgets as widgets
from IPython.display import display

ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    PANEL_DIR  = Path("/content/panels")
    CHECKPOINT = Path("sam_vit_b_01ec64.pth")
    PANEL_ART_AVAILABLE = False
else:
    REPO_ROOT  = Path("../..")
    PANEL_DIR  = REPO_ROOT / "frobenius_artifacts/analysis/panels"
    CHECKPOINT = Path("sam_vit_b_01ec64.pth")
    sys.path.insert(0, str(Path(".").resolve()))
    try:
        from panel_art.motif_segment import (
            filter_and_nms, classify_scale, Detection,
            DEFAULT_IOU_THRESH, DEFAULT_STABILITY_THRESH,
            DEFAULT_NMS_IOU, DEFAULT_MIN_AREA, DEFAULT_MAX_AREA,
            DEFAULT_POINTS_PER_SIDE, DEFAULT_MAX_ASPECT,
        )
        PANEL_ART_AVAILABLE = True
        print("panel_art loaded.")
    except ImportError as e:
        print(f"panel_art not importable ({e}) — using inline helpers.")
        PANEL_ART_AVAILABLE = False

panel_files = sorted(PANEL_DIR.glob("*_panel_*.png")) if PANEL_DIR.exists() else []
print(f"{len(panel_files)} panel crop(s) in {PANEL_DIR}")


In [ ]:
# ── Cell 3: inline helpers (used when panel_art is not importable) ─────────

def _iou(a, b):
    ax1, ay1 = a[0], a[1]; ax2, ay2 = a[0]+a[2], a[1]+a[3]
    bx1, by1 = b[0], b[1]; bx2, by2 = b[0]+b[2], b[1]+b[3]
    iw = max(0, min(ax2,bx2)-max(ax1,bx1))
    ih = max(0, min(ay2,by2)-max(ay1,by1))
    inter = iw * ih
    union = a[2]*a[3] + b[2]*b[3] - inter
    return inter/union if union > 0 else 0.0

def _filter_and_nms(masks, img_area, min_area=0.01, max_area=0.85,
                    iou_thresh=0.0, stability_thresh=0.0,
                    nms_iou=0.40, max_aspect=7.0):
    kept = []
    for m in masks:
        x, y, w, h = m["bbox"]
        ar = m["area"] / img_area
        if ar < min_area or ar > max_area: continue
        if w == 0 or h == 0: continue
        if max(w,h)/max(min(w,h),1) > max_aspect: continue
        if m.get("predicted_iou", 1.0) < iou_thresh: continue
        if m.get("stability_score", 1.0) < stability_thresh: continue
        kept.append(m)
    kept.sort(key=lambda m: m["area"], reverse=True)
    final, suppressed = [], set()
    for i, m in enumerate(kept):
        if i in suppressed: continue
        final.append(m)
        for j in range(i+1, len(kept)):
            if j not in suppressed and _iou(m["bbox"], kept[j]["bbox"]) > nms_iou:
                suppressed.add(j)
    return final

def _classify_scale(area_ratio):
    return "register" if area_ratio > 0.25 else "motif"

if not PANEL_ART_AVAILABLE:
    filter_and_nms  = _filter_and_nms
    classify_scale  = _classify_scale
    DEFAULT_IOU_THRESH       = 0.70
    DEFAULT_STABILITY_THRESH = 0.75
    DEFAULT_NMS_IOU          = 0.40
    DEFAULT_MIN_AREA         = 0.01
    DEFAULT_MAX_AREA         = 0.85
    DEFAULT_POINTS_PER_SIDE  = 32
    DEFAULT_MAX_ASPECT       = 7.0

print("Helpers ready.")


In [ ]:
# ── Cell 4: SAM model loading — automatic generator + point-prompt predictor ──
import torch
from segment_anything import SamAutomaticMaskGenerator, SamPredictor, sam_model_registry

def _resolve_device():
    if torch.cuda.is_available(): return "cuda"
    return "cpu"   # SAM-1 uses float64 — MPS excluded

_sam_model_cache = {}
_predictor_cache = {}

def _load_sam():
    """Load SAM ViT-B once and cache it (shared by both modes)."""
    if "sam" not in _sam_model_cache:
        device = _resolve_device()
        print(f"Loading SAM ViT-B on {device}...")
        sam = sam_model_registry["vit_b"](checkpoint=str(CHECKPOINT))
        sam.to(device=device)
        _sam_model_cache["sam"] = sam
        _sam_model_cache["device"] = device
        print("Done.")
    return _sam_model_cache["sam"]

def get_generator(points_per_side, iou_thresh, stability_thresh):
    """Return a fresh SamAutomaticMaskGenerator (Automatic / v1 mode)."""
    sam = _load_sam()
    return SamAutomaticMaskGenerator(
        model=sam,
        points_per_side=points_per_side,
        pred_iou_thresh=iou_thresh,
        stability_score_thresh=stability_thresh,
        min_mask_region_area=100,
    )

def get_predictor():
    """Return cached SamPredictor (IGSM / point-prompt mode)."""
    if "pred" not in _predictor_cache:
        _predictor_cache["pred"] = SamPredictor(_load_sam())
    return _predictor_cache["pred"]

_ = _load_sam()
print("SAM ready — both automatic and predictor modes available.")

In [20]:
# ── Cell 5: IGSM helpers (binarise → components → point-prompt SAM) ────────
#
# Inspired by Fuentes-Ferrer et al. 2025 (Egyptian hieroglyphs).
# Key difference from automatic SAM:
#   - We run Otsu binarisation first to find *where* to look
#   - SAM is prompted with random pixels inside each connected dark region
#   - This focuses SAM on carved regions, not wood-grain texture
#   - Large woven/knotwork borders appear as large connected components
#     and are naturally included without needing a high area floor

def binarize_otsu(img_np, otsu_adjust=0, morph_kernel=3, morph_iter=2):
    """
    Otsu threshold (inverted) to isolate dark carved/recessed regions.

    Parameters
    ----------
    otsu_adjust : int
        Shift applied to Otsu threshold.  Positive → raise threshold (capture
        less dark area — useful if wood grain is triggering false components).
        Negative → lower threshold (capture more, useful for lightly carved panels).
    morph_kernel : int (odd)
        Structuring element size for open+close morphology.
        Larger = smoother components, fills small gaps in woven patterns.
    morph_iter : int
        Morphology iterations.  2 is a good default.

    Returns
    -------
    binary : H×W uint8 (255 = carved region, 0 = background)
    thresh : int  (actual threshold value used)
    """
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    otsu_val, _ = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    thresh = int(np.clip(otsu_val + otsu_adjust, 1, 254))
    _, binary = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY_INV)

    if morph_kernel >= 1:
        k = morph_kernel + (1 - morph_kernel % 2)   # ensure odd
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  kernel, iterations=morph_iter)
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=morph_iter)

    return binary, thresh


def get_components(binary, min_px=200):
    """
    Connected component analysis on binarised image.

    Parameters
    ----------
    min_px : int
        Minimum component area in pixels.  Small values include fine detail;
        large values focus on complete motifs and border patterns only.

    Returns
    -------
    labels      : H×W int32 label map (0 = background)
    components  : list of dicts with keys: label, area_px, area_ratio, bbox, centroid
    """
    img_h, img_w = binary.shape[:2]
    img_area = img_h * img_w
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )
    components = []
    for i in range(1, num_labels):
        area_px = int(stats[i, cv2.CC_STAT_AREA])
        if area_px < min_px:
            continue
        components.append({
            "label":      i,
            "area_px":    area_px,
            "area_ratio": area_px / img_area,
            "bbox": (
                int(stats[i, cv2.CC_STAT_LEFT]),
                int(stats[i, cv2.CC_STAT_TOP]),
                int(stats[i, cv2.CC_STAT_WIDTH]),
                int(stats[i, cv2.CC_STAT_HEIGHT]),
            ),
            "centroid": (float(centroids[i][0]), float(centroids[i][1])),
        })
    return labels, components


def run_igsm(img_np, predictor, binary, labels, components,
             n_points=8, min_area=0.001, max_area=0.85,
             nms_iou=0.35, max_aspect=5.0, min_score=0.0):
    """
    For each connected component: sample n_points random pixels → SAM point-prompt
    → take best mask → filter + NMS across all resulting masks.

    The SAM image embedding is computed once (expensive), then each predict()
    call (one per component) is fast.

    Returns
    -------
    raw_masks : list of all SAM mask dicts before NMS
    kept      : list of masks after filter_and_nms
    """
    img_h, img_w = img_np.shape[:2]
    img_area = img_h * img_w
    rng = np.random.default_rng(42)   # reproducible point sampling

    predictor.set_image(img_np)       # encode image once

    raw_masks = []
    for comp in components:
        pixel_yx = np.argwhere(labels == comp["label"])   # shape (N, 2): row, col
        n_pts = min(n_points, len(pixel_yx))
        if n_pts < 1:
            continue

        chosen = pixel_yx[rng.choice(len(pixel_yx), n_pts, replace=False)]
        input_points = chosen[:, ::-1].astype(float)      # → (x, y)
        input_labels_arr = np.ones(n_pts, dtype=int)      # all foreground

        try:
            masks, scores, _ = predictor.predict(
                point_coords=input_points,
                point_labels=input_labels_arr,
                multimask_output=True,
            )
        except Exception:
            continue

        best = int(np.argmax(scores))
        mask = masks[best]
        score = float(scores[best])

        rows_on = np.where(np.any(mask, axis=1))[0]
        cols_on = np.where(np.any(mask, axis=0))[0]
        if len(rows_on) == 0 or len(cols_on) == 0:
            continue

        rmin, rmax = int(rows_on[0]),  int(rows_on[-1])
        cmin, cmax = int(cols_on[0]),  int(cols_on[-1])

        raw_masks.append({
            "segmentation":   mask,
            "area":           int(mask.sum()),
            "bbox":           [cmin, rmin, cmax - cmin, rmax - rmin],
            "predicted_iou":  score,
            "stability_score": score,   # predictor has no separate stability
            "_component":     comp,
        })

    kept = filter_and_nms(
        raw_masks, img_area,
        min_area=min_area, max_area=max_area,
        iou_thresh=min_score, stability_thresh=0.0,
        nms_iou=nms_iou, max_aspect=max_aspect,
    )
    return raw_masks, kept

print("IGSM helpers ready.")

IGSM helpers ready.


In [21]:
# ── Cell 6: drawing helpers ────────────────────────────────────────────────

SCALE_COLOURS = {
    "register": (1.0, 0.31, 0.31),   # red
    "motif":    (0.31, 0.78, 0.31),  # green
}

def _det_fields(d):
    """Unpack a Detection object or dict uniformly."""
    if PANEL_ART_AVAILABLE and isinstance(d, Detection):
        b = d.bbox
        return b["x"], b["y"], b["w"], b["h"], d.scale, d.area_ratio, d.index, d.predicted_iou
    b = d["bbox"]
    return b[0], b[1], b[2], b[3], d["scale"], d["area_ratio"], d["index"], d["predicted_iou"]


def draw_detections(img_rgb, detections, ax, title=""):
    ax.imshow(img_rgb)
    h, w = img_rgb.shape[:2]
    fs = max(5, min(w, h) / 80)
    lw = max(1, min(w, h) // 300)
    for d in detections:
        bx, by, bw, bh, scale, ar, idx, _ = _det_fields(d)
        colour = SCALE_COLOURS.get(scale, (0.8, 0.8, 0.8))
        ax.add_patch(mpatches.Rectangle((bx, by), bw, bh,
            linewidth=lw, edgecolor=colour, facecolor="none"))
        ax.text(bx+2, by-3, f"#{idx} {scale} {ar*100:.2f}%",
            fontsize=fs, color="white",
            bbox=dict(facecolor=colour, edgecolor="none", pad=1, alpha=0.85),
            va="bottom")
    ax.legend(handles=[mpatches.Patch(color=c, label=s) for s,c in SCALE_COLOURS.items()],
              loc="upper right", fontsize=fs)
    ax.axis("off")
    if title: ax.set_title(title, fontsize=10)


def draw_binary_overlay(img_rgb, binary, components, ax, title="Binarisation"):
    """
    Tint binarised pixels blue, draw cyan bounding boxes per component.
    Useful for tuning otsu_adjust and morph_kernel before running SAM.
    """
    overlay = img_rgb.copy().astype(float) / 255.0
    mask_bool = binary > 0
    # Blue tint on dark/carved regions
    overlay[mask_bool] = overlay[mask_bool] * 0.45 + np.array([0.0, 0.35, 1.0]) * 0.55
    ax.imshow(np.clip(overlay, 0, 1))
    h, w = img_rgb.shape[:2]
    fs = max(4, min(w, h) / 130)
    lw = max(1, min(w, h) // 500)
    for comp in components:
        x, y, bw, bh = comp["bbox"]
        ax.add_patch(mpatches.Rectangle((x, y), bw, bh,
            linewidth=lw, edgecolor="cyan", facecolor="none", alpha=0.9))
        ax.text(x+2, y+fs+2, f"{comp['area_ratio']*100:.2f}%",
            fontsize=fs-1, color="cyan", va="top")
    ax.axis("off")
    ax.set_title(title, fontsize=9)


def draw_raw_igsm(img_rgb, raw_masks, kept_masks, img_area, ax,
                  mina, maxa, masp):
    """Colour-code all raw IGSM masks by drop reason (like v1 raw view)."""
    ax.imshow(img_rgb)
    h, w = img_rgb.shape[:2]
    fs = max(4, min(w, h) / 130)
    lw = max(1, min(w, h) // 400)

    kept_bboxes = {tuple(m["bbox"]) for m in kept_masks}

    for m in raw_masks:
        x, y, bw, bh = [int(v) for v in m["bbox"]]
        ar = m["area"] / img_area
        asp = max(bw, bh) / max(min(bw, bh), 1) if min(bw, bh) > 0 else 999

        if tuple(m["bbox"]) in kept_bboxes:
            colour, reason = "lime", "kept"
        elif ar < mina:
            colour, reason = "yellow", f"small {ar*100:.2f}%"
        elif ar > maxa:
            colour, reason = "orange", f"large {ar*100:.1f}%"
        elif asp > masp:
            colour, reason = "mediumpurple", f"aspect {asp:.1f}"
        else:
            colour, reason = "lightgrey", "NMS"

        ax.add_patch(mpatches.Rectangle((x,y), bw, bh,
            linewidth=lw, edgecolor=colour, facecolor="none", alpha=0.7))
        ax.text(x+2, y+10, reason, fontsize=fs-1, color=colour, va="top", alpha=0.9)

    ax.legend(handles=[
        mpatches.Patch(color="lime",         label="kept"),
        mpatches.Patch(color="yellow",       label="too small"),
        mpatches.Patch(color="orange",       label="too large"),
        mpatches.Patch(color="mediumpurple", label="bad aspect"),
        mpatches.Patch(color="lightgrey",    label="NMS"),
    ], loc="upper right", fontsize=fs, framealpha=0.8)
    ax.set_title(f"All {len(raw_masks)} raw IGSM masks", fontsize=9)
    ax.axis("off")

print("Drawing helpers ready.")

Drawing helpers ready.


In [24]:
# ── Cell 7: spatial encoding & reading-order grid ──────────────────────────
#
# Goal: assign each detection a (row, col) position so motifs can be
# described by their relative spatial relationships — a first step toward
# encoding symbol sequences for later classification/retrieval.

def compute_spatial_positions(detections, img_h, img_w):
    """
    Return list of (cy_norm, cx_norm, detection) sorted top→bottom, left→right.
    Normalised coordinates (0–1) make positions comparable across panel sizes.
    """
    out = []
    for d in detections:
        if PANEL_ART_AVAILABLE and isinstance(d, Detection):
            b = d.bbox
            cx = (b["x"] + b["w"] / 2) / img_w
            cy = (b["y"] + b["h"] / 2) / img_h
        else:
            b = d["bbox"]
            cx = (b[0] + b[2] / 2) / img_w
            cy = (b[1] + b[3] / 2) / img_h
        out.append((cy, cx, d))
    out.sort(key=lambda t: (t[0], t[1]))
    return out


def draw_spatial_grid(detections, img_h, img_w, ax, n_rows=10, n_cols=5):
    """
    Schematic grid: each cell coloured by scale, labelled with detection index.
    Designed for tall vertical panels (n_rows > n_cols).

    The grid represents reading order top→bottom, left→right within each row.
    Multiple detections in the same cell are shown stacked.
    """
    # Collect detections per cell (allow multiples)
    from collections import defaultdict
    cell_dets = defaultdict(list)

    for d in detections:
        if PANEL_ART_AVAILABLE and isinstance(d, Detection):
            b = d.bbox; idx = d.index; scale = d.scale
            cx = b["x"] + b["w"] / 2; cy = b["y"] + b["h"] / 2
        else:
            b = d["bbox"]; idx = d["index"]; scale = d["scale"]
            cx = b[0] + b[2] / 2; cy = b[1] + b[3] / 2
        row = int(min(cy / img_h * n_rows, n_rows - 1))
        col = int(min(cx / img_w * n_cols, n_cols - 1))
        cell_dets[(row, col)].append((idx, scale))

    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(n_rows - 0.5, -0.5)
    ax.set_aspect("equal")
    ax.set_xticks(range(n_cols)); ax.set_yticks(range(n_rows))
    ax.set_xticklabels([f"C{i}" for i in range(n_cols)], fontsize=7)
    ax.set_yticklabels([f"R{i}" for i in range(n_rows)], fontsize=7)
    ax.grid(True, color="grey", linewidth=0.4, alpha=0.4)
    ax.set_facecolor("#1a1a1a")

    for (row, col), dets in cell_dets.items():
        # Use scale of first detection for colour
        colour = SCALE_COLOURS.get(dets[0][1], (0.8, 0.8, 0.8))
        n = len(dets)
        ax.add_patch(mpatches.Rectangle(
            (col - 0.45, row - 0.45), 0.9, 0.9,
            facecolor=colour, edgecolor="white", linewidth=0.5, alpha=0.75,
        ))
        label = ",".join(str(idx) for idx, _ in dets[:4])
        if len(dets) > 4: label += "…"
        ax.text(col, row, label, ha="center", va="center",
                fontsize=7, fontweight="bold", color="white")

    ax.set_title(
        f"Spatial grid {n_rows}R×{n_cols}C — reading order T→B L→R"
        f"({len(detections)} detections)",
        fontsize=9,
    )

print("Spatial helpers ready.")

Spatial helpers ready.


In [ ]:
# ── Cell 8: interactive tuning UI (v2) ─────────────────────────────────────

# ── Panel selector ────────────────────────────────────────────────────────────
panel_picker = widgets.SelectMultiple(
    options=[(p.name, str(p)) for p in panel_files],
    value=[str(panel_files[0])] if panel_files else [],
    rows=8,
    description="Panels:",
    layout=widgets.Layout(width="70%"),
    style={"description_width": "60px"},
)
w_sel_count = widgets.HTML(value="")
out_preview = widgets.Output()

def _show_previews(paths):
    out_preview.clear_output(wait=True)
    w_sel_count.value = (
        f"<i style='color:#666'>{len(paths)} selected — "
        f"Ctrl/⌘+click to add, Shift+click to range-select</i>"
    )
    if not paths:
        return
    with out_preview:
        n = len(paths)
        thumb_w = min(3.5, 18 / max(n, 1))
        fig, axes = plt.subplots(1, n, figsize=(thumb_w * n, 3.5), squeeze=False)
        for ax, p in zip(axes[0], paths):
            img = Image.open(p).convert("RGB")
            img.thumbnail((300, 380))
            ax.imshow(np.array(img))
            ax.set_title(Path(p).name, fontsize=6)
            ax.axis("off")
        plt.tight_layout(pad=0.3)
        plt.show()

def _on_picker_change(change):
    _show_previews(list(change["new"]))

panel_picker.observe(_on_picker_change, names="value")
_show_previews(list(panel_picker.value))

# ── Mode ──────────────────────────────────────────────────────────────────────
w_mode = widgets.ToggleButtons(
    options=["Automatic (v1)", "Focused IGSM", "Compare both"],
    value="Focused IGSM",
    description="Mode:",
    style={"description_width": "50px", "button_width": "140px"},
)

# ── Shared style for all sliders ──────────────────────────────────────────────
_sl = dict(continuous_update=False, style={"description_width": "120px"},
           layout=widgets.Layout(width="90%"))

# ── Automatic (v1) params ─────────────────────────────────────────────────────
w_points = widgets.SelectionSlider(
    options=[8, 16, 32, 64], value=DEFAULT_POINTS_PER_SIDE,
    description="points/side", **_sl,
)
w_iou_thresh = widgets.FloatSlider(
    min=0.40, max=0.95, step=0.01, value=DEFAULT_IOU_THRESH,
    description="iou_thresh", readout_format=".2f", **_sl,
)
w_stab = widgets.FloatSlider(
    min=0.40, max=0.95, step=0.01, value=DEFAULT_STABILITY_THRESH,
    description="stability", readout_format=".2f", **_sl,
)

# ── IGSM params ───────────────────────────────────────────────────────────────
w_otsu_adjust = widgets.IntSlider(
    min=-80, max=80, step=2, value=0,
    description="otsu_adjust", **_sl,
)
w_morph_kernel = widgets.IntSlider(
    min=1, max=11, step=2, value=3,
    description="morph_kernel", **_sl,
)
w_morph_iter = widgets.IntSlider(
    min=1, max=5, step=1, value=2,
    description="morph_iter", **_sl,
)
w_min_comp_px = widgets.IntSlider(
    min=50, max=15000, step=50, value=500,
    description="min_comp_px", **_sl,
)
w_n_points = widgets.IntSlider(
    min=3, max=15, step=1, value=8,
    description="n_points", **_sl,
)
w_min_score = widgets.FloatSlider(
    min=0.0, max=0.90, step=0.05, value=0.50,
    description="min_score", readout_format=".2f", **_sl,
)

# ── Shared filter params (area in %, divided by 100 before use) ───────────────
w_min_area = widgets.FloatSlider(
    min=0.05, max=15.0, step=0.05, value=DEFAULT_MIN_AREA * 100,
    description="min_area %", readout_format=".2f", **_sl,
)
w_max_area = widgets.FloatSlider(
    min=30.0, max=100.0, step=1.0, value=DEFAULT_MAX_AREA * 100,
    description="max_area %", readout_format=".0f", **_sl,
)
w_nms_iou = widgets.FloatSlider(
    min=0.05, max=0.70, step=0.01, value=DEFAULT_NMS_IOU,
    description="nms_iou", readout_format=".2f", **_sl,
)
w_max_aspect = widgets.FloatSlider(
    min=1.5, max=15.0, step=0.5, value=DEFAULT_MAX_ASPECT,
    description="max_aspect", readout_format=".1f", **_sl,
)

# ── Display toggles ───────────────────────────────────────────────────────────
w_show_binary  = widgets.Checkbox(value=True,
    description="Show binarisation + components",
    style={"description_width": "initial"}, layout=widgets.Layout(width="50%"))
w_show_raw     = widgets.Checkbox(value=False,
    description="Show all raw masks (colour-coded drop reason)",
    style={"description_width": "initial"}, layout=widgets.Layout(width="50%"))
w_show_spatial = widgets.Checkbox(value=True,
    description="Show spatial grid (reading-order positions)",
    style={"description_width": "initial"}, layout=widgets.Layout(width="50%"))

run_btn     = widgets.Button(description="Run segmentation", button_style="primary",
                             layout=widgets.Layout(width="200px", height="36px"))
out_status  = widgets.Output()
out_binary  = widgets.Output()
out_image   = widgets.Output()
out_raw     = widgets.Output()
out_table   = widgets.Output()
out_spatial = widgets.Output()
out_export  = widgets.Output()

# ── Layout ────────────────────────────────────────────────────────────────────
auto_box = widgets.VBox([
    widgets.HTML("<b>Auto (v1) parameters</b>"),
    w_points, w_iou_thresh, w_stab,
], layout=widgets.Layout(border="1px solid #aaa", padding="8px",
                         width="48%", margin="0 1% 0 0"))

igsm_box = widgets.VBox([
    widgets.HTML("<b>IGSM parameters</b>"),
    w_otsu_adjust, w_morph_kernel, w_morph_iter,
    w_min_comp_px, w_n_points, w_min_score,
], layout=widgets.Layout(border="1px solid #aaa", padding="8px", width="48%"))

shared_box = widgets.VBox([
    widgets.HTML("<b>Shared filter parameters</b>"),
    w_min_area, w_max_area, w_nms_iou, w_max_aspect,
], layout=widgets.Layout(border="1px solid #ddd", padding="8px", margin="6px 0"))

display(
    widgets.HTML("<h3 style='margin-bottom:4px'>SAM Motif Segmentation — v2</h3>"),
    panel_picker,
    w_sel_count,
    out_preview,
    w_mode,
    widgets.HBox([auto_box, igsm_box]),
    shared_box,
    widgets.HBox([w_show_binary, w_show_raw, w_show_spatial]),
    run_btn,
    out_status, out_binary, out_image, out_raw,
    out_table, out_spatial, out_export,
)

# ── Helper: build Detection list from kept masks ──────────────────────────────
def _build_detections(kept, img_area):
    dets = []
    for idx, m in enumerate(kept):
        x, y, bw, bh = [int(v) for v in m["bbox"]]
        ar = m["area"] / img_area
        if PANEL_ART_AVAILABLE:
            dets.append(Detection(
                index=idx, bbox={"x": x, "y": y, "w": bw, "h": bh},
                scale=classify_scale(ar), area_ratio=ar,
                predicted_iou=float(m["predicted_iou"]),
                stability_score=float(m.get("stability_score", m["predicted_iou"])),
            ))
        else:
            dets.append({"index": idx, "bbox": [x, y, bw, bh],
                "scale": classify_scale(ar), "area_ratio": ar,
                "predicted_iou": float(m["predicted_iou"]),
                "stability_score": float(m.get("stability_score", m["predicted_iou"])),
            })
    key = (lambda d: d.area_ratio) if PANEL_ART_AVAILABLE else (lambda d: d["area_ratio"])
    dets.sort(key=key, reverse=True)
    for i, d in enumerate(dets):
        if PANEL_ART_AVAILABLE: d.index = i
        else: d["index"] = i
    return dets


# ── Run callback ──────────────────────────────────────────────────────────────
def on_run(_):
    for out in [out_status, out_binary, out_image, out_raw,
                out_table, out_spatial, out_export]:
        out.clear_output(wait=True)

    selected = [Path(p) for p in panel_picker.value]
    if not selected:
        with out_status:
            print("No panels selected — Ctrl/⌘+click to select one or more.")
        return

    mode      = w_mode.value
    pts       = int(w_points.value)
    iou_t     = float(w_iou_thresh.value)
    stab      = float(w_stab.value)
    otsu_adj  = int(w_otsu_adjust.value)
    morph_k   = int(w_morph_kernel.value)
    morph_it  = int(w_morph_iter.value)
    min_comp  = int(w_min_comp_px.value)
    n_pts     = int(w_n_points.value)
    min_score = float(w_min_score.value)
    mina      = float(w_min_area.value) / 100   # widget shows %, filter expects fraction
    maxa      = float(w_max_area.value) / 100
    nms       = float(w_nms_iou.value)
    masp      = float(w_max_aspect.value)

    with out_status:
        print(f"Mode: {mode}  |  {len(selected)} panel(s)\n")

    for panel_path in selected:
        with out_status:
            print(f"▶ {panel_path.name}")

        img_pil  = Image.open(panel_path).convert("RGB")
        img_np   = np.array(img_pil)
        h, w     = img_np.shape[:2]
        img_area = h * w

        auto_dets  = None
        igsm_dets  = None
        igsm_raw   = None
        kept_igsm  = None
        binary     = None
        components = None

        # ── Automatic (v1) ────────────────────────────────────────────────
        if mode in ("Automatic (v1)", "Compare both"):
            with out_status: print("  Running Automatic grid SAM...")
            gen = get_generator(pts, iou_t, stab)
            auto_raw_masks = gen.generate(img_np)
            kept_auto = filter_and_nms(auto_raw_masks, img_area,
                min_area=mina, max_area=maxa, iou_thresh=iou_t,
                stability_thresh=stab, nms_iou=nms, max_aspect=masp)
            auto_dets = _build_detections(kept_auto, img_area)
            with out_status:
                print(f"  Auto: {len(auto_raw_masks)} raw → {len(auto_dets)} kept")

        # ── IGSM ──────────────────────────────────────────────────────────
        if mode in ("Focused IGSM", "Compare both"):
            with out_status: print("  Running Focused IGSM...")
            binary, thresh = binarize_otsu(img_np, otsu_adj, morph_k, morph_it)
            labels, components = get_components(binary, min_px=min_comp)
            predictor = get_predictor()
            igsm_raw, kept_igsm = run_igsm(
                img_np, predictor, binary, labels, components,
                n_points=n_pts, min_area=mina, max_area=maxa,
                nms_iou=nms, max_aspect=masp, min_score=min_score,
            )
            igsm_dets = _build_detections(kept_igsm, img_area)
            with out_status:
                print(f"  IGSM: {len(components)} components, "
                      f"{len(igsm_raw)} raw → {len(igsm_dets)} kept "
                      f"(Otsu thresh={thresh})")

        # ── Binarisation overlay ───────────────────────────────────────────
        if w_show_binary.value and binary is not None:
            with out_binary:
                fh = min(16, max(6, h/80)); fw = fh * w/h
                fig, ax = plt.subplots(figsize=(fw, fh))
                draw_binary_overlay(img_np, binary, components, ax,
                    title=f"{panel_path.name} — Binarisation "
                          f"(Otsu thresh={thresh}, {len(components)} components ≥ {min_comp}px)")
                plt.tight_layout(); plt.show()

        # ── Detection image(s) ─────────────────────────────────────────────
        with out_image:
            fh = min(20, max(8, h/80)); fw = fh * w/h
            if mode == "Compare both":
                fig, axes = plt.subplots(1, 2, figsize=(fw*2, fh))
                draw_detections(img_np, auto_dets, axes[0],
                    title=f"{panel_path.name} — Auto ({len(auto_dets)} detections)")
                draw_detections(img_np, igsm_dets, axes[1],
                    title=f"{panel_path.name} — IGSM ({len(igsm_dets)} detections)")
            else:
                fig, ax = plt.subplots(figsize=(fw, fh))
                dets = auto_dets if mode == "Automatic (v1)" else igsm_dets
                draw_detections(img_np, dets, ax,
                    title=f"{panel_path.name}  |  {mode}  |  {len(dets)} detections")
            plt.tight_layout(); plt.show()

        # ── Raw mask view ──────────────────────────────────────────────────
        if w_show_raw.value and igsm_raw is not None:
            with out_raw:
                fh = min(20, max(8, h/80)); fw = fh * w/h
                fig, ax = plt.subplots(figsize=(fw, fh))
                draw_raw_igsm(img_np, igsm_raw, kept_igsm, img_area, ax, mina, maxa, masp)
                ax.set_title(f"{panel_path.name} — {len(igsm_raw)} raw IGSM masks", fontsize=9)
                plt.tight_layout(); plt.show()

        # ── Detection table ────────────────────────────────────────────────
        primary = igsm_dets if igsm_dets is not None else auto_dets
        if primary:
            positioned = compute_spatial_positions(primary, h, w)
            with out_table:
                print(f"\n── {panel_path.name} ──────────────────────────────────────")
                print(f"{'#':>3}  {'scale':<10} {'area%':>6}  "
                      f"{'bbox (x,y,w,h)':>26}  {'score':>6}  {'pos(y,x)':>10}")
                print("-" * 72)
                for cy, cx, d in positioned:
                    bx, by, bw, bh, scale, ar, idx, score = _det_fields(d)
                    print(f"{idx:>3}  {scale:<10} {ar*100:>5.2f}%  "
                          f"({bx:>4},{by:>4},{bw:>4},{bh:>4})  "
                          f"{score:>6.3f}  ({cy:.2f},{cx:.2f})")

            if w_show_spatial.value:
                with out_spatial:
                    fig_s, ax_s = plt.subplots(figsize=(5, 9))
                    draw_spatial_grid(primary, h, w, ax_s, n_rows=10, n_cols=5)
                    ax_s.set_title(
                        f"{panel_path.name}\n"
                        f"Spatial grid 10R×5C — reading order T→B L→R\n"
                        f"({len(primary)} detections)",
                        fontsize=9,
                    )
                    plt.tight_layout(); plt.show()

        with out_status:
            print()   # blank line between panels

    # ── Shared parameter export (once, after all panels) ──────────────────
    with out_export:
        print("\n# ── Parameter snapshot ─────────────────────────────")
        print(f"# Mode: {mode}  |  panels: {len(selected)}")
        print(f"DEFAULT_MIN_AREA   = {mina}")
        print(f"DEFAULT_MAX_AREA   = {maxa}")
        print(f"DEFAULT_NMS_IOU    = {nms}")
        print(f"DEFAULT_MAX_ASPECT = {masp}")
        if mode != "Focused IGSM":
            print(f"DEFAULT_IOU_THRESH       = {iou_t}")
            print(f"DEFAULT_STABILITY_THRESH = {stab}")
            print(f"DEFAULT_POINTS_PER_SIDE  = {pts}")
        if mode != "Automatic (v1)":
            print(f"IGSM_OTSU_ADJUST      = {otsu_adj}")
            print(f"IGSM_MORPH_KERNEL     = {morph_k}")
            print(f"IGSM_MORPH_ITER       = {morph_it}")
            print(f"IGSM_MIN_COMPONENT_PX = {min_comp}")
            print(f"IGSM_N_POINTS         = {n_pts}")
            print(f"IGSM_MIN_SCORE        = {min_score}")


run_btn.on_click(on_run)


---
## Parameter guide — v2

### Mode selector

| Mode | When to use |
|---|---|
| **Automatic (v1)** | Baseline — same as v1 notebook. Uniform SAM grid across the whole image. |
| **Focused IGSM** | Better for dense carved panels. SAM is only prompted inside dark binarised regions. Also captures woven border patterns naturally. |
| **Compare both** | Side-by-side comparison to see what each mode finds that the other misses. |

---

### IGSM parameters

**`otsu_adjust`** (−80 → +80, default 0)
Shifts the Otsu threshold up (positive) or down (negative).
- **Positive**: only very dark carved recesses trigger components — fewer, cleaner regions.
- **Negative**: lighter carved edges also trigger — more regions, including subtle surface relief.
Start at 0, move negative if you are missing shallow engravings, move positive if wood grain
is generating spurious components.

**`morph_kernel`** (1–11 odd, default 3)
Structuring element size for the open+close morphology step.
Larger values merge nearby dark pixels into solid components — essential for woven/knotwork
patterns whose carved lines are thin but form a large continuous dark region.
Try 7–11 if border patterns are fragmenting into many small components.

**`morph_iter`** (1–5, default 2)
How many times the morphological operations are applied. More iterations = smoother, more
connected components. Rarely needs changing.

**`min_comp_px`** (50–15000, default 500)
Minimum component size in **pixels** (not a fraction). Small values include fine scratches and
noise. Large values focus SAM on whole motifs and complete border patterns only.
On a 1000×1500 panel, 500px ≈ 0.03% of area; 5000px ≈ 0.33%.

**`n_points`** (3–15, default 8)
Random SAM prompt points per component. The paper uses 8–11 for carved stone, 3–8 for
painted stone. More points help SAM cover irregular or fragmented carved regions, but increase
inference time linearly.

**`min_score`** (0.0–0.90, default 0.50)
SAM predicted-IOU quality floor for IGSM masks. Lower to admit uncertain masks; raise if
you are getting many spurious detections from low-confidence SAM outputs.

---

### Shared parameters

**`min_area`** / **`max_area`**: same meaning as v1 — fraction of panel area.
In IGSM mode, woven border patterns naturally appear as large components and large masks;
raise `max_area` to 0.95+ to include full-panel border patterns.

**`nms_iou`**: IoU threshold for non-maximum suppression across all masks.

**`max_aspect`**: drop masks whose bounding box is more elongated than this ratio.

---

### Spatial grid

The 10×5 grid (10 rows, 5 columns) maps detected motifs to approximate reading-order
positions. Detections are sorted top→bottom, left→right. This grid is the starting point
for encoding relative positional relationships between symbol classes.